# Development-only baseline backtesting

This notebook reproduces the frozen persistence and annual seasonal-persistence baselines across 54 development origins and horizons 1-4. It verifies forecast coverage, WAPE, quarterly seasonal MASE, signed error, origin/state diagnostics, and development-only volume bands. It does not evaluate a candidate model or open holdout performance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Use the same private Drive root as notebooks 01 and 02. This run validates the source checkpoint lineage and writes versioned baseline artifacts beneath `runs/`.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ds_portfolio/project_02_coal_production_forecasting')
DATA_ROOT = DRIVE_PROJECT_ROOT / 'data'
RUNS_ROOT = DRIVE_PROJECT_ROOT / 'runs'

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/ahmaddshbg-blip/regional-coal-production-forecasting.git'
REPO_DIR = Path('/content/regional-coal-production-forecasting')
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-lock.txt')],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR), '--no-deps'],
    check=True,
)
source_root = str(REPO_DIR / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)
revision = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
print('Code revision:', revision)

In [ ]:
import os

os.chdir(REPO_DIR)
os.environ['PROJECT_DATA_ROOT'] = str(DATA_ROOT)
os.environ['PROJECT_RUNS_ROOT'] = str(RUNS_ROOT)
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=REPO_DIR, check=True
)

## Frozen evaluation boundary

The runner accepts only the configured development origins. Its latest target is derived as `2021Q1 + 4 quarters = 2022Q1`, which precedes the first holdout origin, `2022Q2`. Any later target value or incomplete baseline prediction causes the run to fail.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter

from coal_forecasting.evaluation import (
    FROZEN_BASELINE_AUDIT,
    run_development_baseline_evaluation,
)

result = run_development_baseline_evaluation(
    REPO_DIR / 'configs' / 'project.json',
    root=REPO_DIR,
)
manifest = result['manifest']
forecasts = result['forecasts']
metrics = result['metrics']
state_metrics = result['state_metrics']
origin_metrics = result['origin_metrics']
volume_metrics = result['volume_metrics']
print('Baseline run:', manifest['run_id'])
print('Manifest:', manifest['manifest_path'])

In [ ]:
manifest_summary = pd.Series(
    {
        'stage': manifest['stage'],
        'status': manifest['status'],
        'source data run': manifest['source_data_run'],
        'target end': manifest['parameters']['target_end'],
        'git commit': manifest['git']['commit'],
        'dirty worktree': manifest['git']['dirty'],
        'holdout opened': manifest['checks']['holdout_opened'],
    },
    name='value',
)
display(manifest_summary.to_frame())
display(
    pd.DataFrame(manifest['outputs']).T[
        ['path', 'rows', 'bytes', 'sha256']
    ]
)

## Cell coverage

Each method must forecast every eligible state-origin-horizon cell. Scoring availability may be slightly lower because a future actual can be absent; those actuals remain missing rather than being replaced by zero.

In [ ]:
coverage_view = metrics[
    ['model', 'horizon', 'forecast_rows', 'predicted_rows', 'scored_cells',
     'prediction_coverage', 'target_availability', 'defined_mase_cells',
     'defined_mase_states']
].copy()
coverage_view['prediction_coverage'] *= 100
coverage_view['target_availability'] *= 100
assert coverage_view['prediction_coverage'].eq(100).all()
assert forecasts['origin_date'].max() == pd.Timestamp('2021-01-01')
assert forecasts['target_date'].max() == pd.Timestamp('2022-01-01')
display(coverage_view.round(3))

## Reproduction of the frozen Gate 2 audit

The exact scored-cell counts and the rounded WAPE and median state MASE below were recorded before this implementation. The run fails inside the reusable evaluator if it does not reproduce them.

In [ ]:
expected_audit = pd.DataFrame.from_records(
    [
        {
            'model': model,
            'horizon': horizon,
            'expected_scored_cells': expected[0],
            'expected_wape_percent': expected[1],
            'expected_median_state_mase': expected[2],
        }
        for (model, horizon), expected in FROZEN_BASELINE_AUDIT.items()
    ]
)
audit = metrics.merge(expected_audit, on=['model', 'horizon'], validate='one_to_one')
audit['observed_wape_percent'] = (audit['wape'] * 100).round(2)
audit['observed_median_state_mase'] = audit['median_state_mase'].round(3)
audit['audit_passed'] = (
    audit['scored_cells'].eq(audit['expected_scored_cells'])
    & audit['observed_wape_percent'].eq(audit['expected_wape_percent'])
    & audit['observed_median_state_mase'].eq(audit['expected_median_state_mase'])
)
assert audit['audit_passed'].all()
display(
    audit[
        ['model', 'horizon', 'scored_cells', 'observed_wape_percent',
         'observed_median_state_mase', 'audit_passed']
    ]
)

In [ ]:
colors = {'persistence': '#2563eb', 'seasonal_naive': '#d97706'}
labels = {'persistence': 'Persistence', 'seasonal_naive': 'Annual seasonal persistence'}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for model in ['persistence', 'seasonal_naive']:
    subset = metrics.loc[metrics['model'].eq(model)]
    axes[0].plot(subset['horizon'], subset['wape'], marker='o', linewidth=2, color=colors[model], label=labels[model])
    axes[1].plot(subset['horizon'], subset['median_state_mase'], marker='o', linewidth=2, color=colors[model], label=labels[model])
axes[0].set(title='Development WAPE by horizon', xlabel='Forecast horizon', ylabel='WAPE', xticks=[1, 2, 3, 4])
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].set(title='Median state seasonal MASE', xlabel='Forecast horizon', ylabel='Median state MASE', xticks=[1, 2, 3, 4])
for ax in axes:
    ax.grid(axis='y', alpha=0.2)
    ax.legend(frameon=False)
fig.tight_layout()

## Error magnitude and signed bias

Signed error is `forecast - actual`; positive values indicate overforecasting. WAPE remains the primary volume metric, while MAE, RMSE, and signed bias expose scale and direction.

In [ ]:
error_view = metrics[
    ['model', 'horizon', 'mae', 'rmse', 'signed_mean_error',
     'aggregate_signed_bias', 'mean_state_mase', 'median_state_mase']
].copy()
for column in ['mae', 'rmse', 'signed_mean_error']:
    error_view[column] = error_view[column] / 1_000_000
error_view['aggregate_signed_bias_percent'] = error_view.pop('aggregate_signed_bias') * 100
display(error_view.round(3))

## Origin, state, and volume diagnostics

Aggregate scores can conceal regime shocks and regional imbalance. The following views remain within development origins and do not alter either comparator.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for horizon, ax in zip([1, 2, 3, 4], axes.flat):
    for model in ['persistence', 'seasonal_naive']:
        subset = origin_metrics.loc[
            origin_metrics['model'].eq(model) & origin_metrics['horizon'].eq(horizon)
        ]
        ax.plot(subset['origin_date'], subset['wape'], color=colors[model], linewidth=1.4, label=labels[model])
    ax.set_title(f'Horizon {horizon}')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.grid(axis='y', alpha=0.2)
axes[0, 0].legend(frameon=False)
fig.suptitle('WAPE by development forecast origin', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])

In [ ]:
state_average = state_metrics.groupby(['model', 'state_code'], as_index=False).agg(
    mean_mase=('mase', 'mean'),
    mean_actual_volume=('actual_volume', 'mean'),
)
state_comparison = state_average.pivot(index='state_code', columns='model', values='mean_mase').dropna()
state_volume = state_average.loc[state_average['model'].eq('persistence')].set_index('state_code')['mean_actual_volume']
state_comparison['mean_actual_volume'] = state_volume
log_volume = np.log1p(state_comparison['mean_actual_volume'])
marker_size = 30 + 120 * (log_volume - log_volume.min()) / (log_volume.max() - log_volume.min())
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(state_comparison['persistence'], state_comparison['seasonal_naive'], s=marker_size, color='#0f766e', alpha=0.75)
limit = max(state_comparison['persistence'].max(), state_comparison['seasonal_naive'].max()) * 1.05
ax.plot([0, limit], [0, limit], color='#6b7280', linestyle='--', linewidth=1)
for state in state_comparison.nlargest(8, 'mean_actual_volume').index:
    ax.annotate(state, (state_comparison.loc[state, 'persistence'], state_comparison.loc[state, 'seasonal_naive']), xytext=(4, 4), textcoords='offset points')
ax.set(title='State-balanced baseline comparison', xlabel='Persistence: mean MASE across horizons', ylabel='Seasonal: mean MASE across horizons', xlim=(0, limit), ylim=(0, limit))
ax.grid(alpha=0.2)
fig.tight_layout()
display(state_average.sort_values(['model', 'mean_mase'], ascending=[True, False]).head(16).round(3))

In [ ]:
band_order = ['low', 'medium', 'high', 'very_high']
display(
    volume_metrics.assign(
        wape_percent=volume_metrics['wape'] * 100,
        signed_bias_percent=volume_metrics['aggregate_signed_bias'] * 100,
    )[
        ['model', 'horizon', 'actual_volume_band', 'scored_cells',
         'wape_percent', 'signed_bias_percent']
    ].round(2)
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
band_colors = {'low': '#dc2626', 'medium': '#d97706', 'high': '#0f766e', 'very_high': '#2563eb'}
for ax, model in zip(axes, ['persistence', 'seasonal_naive']):
    for band in band_order:
        subset = volume_metrics.loc[
            volume_metrics['model'].eq(model) & volume_metrics['actual_volume_band'].eq(band)
        ]
        ax.plot(subset['horizon'], subset['wape'], marker='o', color=band_colors[band], label=band.replace('_', ' ').title())
    ax.set(title=labels[model], xlabel='Forecast horizon', xticks=[1, 2, 3, 4])
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.grid(axis='y', alpha=0.2)
axes[0].set_ylabel('WAPE')
axes[1].legend(frameon=False, title='Actual volume band')
fig.suptitle('Development error by actual-volume quartile', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])

## Frozen comparator and readiness

The comparator remains persistence at horizons 1-2, annual seasonal persistence at horizon 3, and their common horizon-4 forecast. The near ties at horizons 2-3 are not evidence that either rule is universally superior.

In [ ]:
wape_comparison = metrics.pivot(index='horizon', columns='model', values='wape')
comparator = pd.DataFrame(
    {
        'horizon': [1, 2, 3, 4],
        'frozen_primary_comparator': ['persistence', 'persistence', 'seasonal_naive', 'common_h4'],
        'persistence_wape_percent': (wape_comparison['persistence'] * 100).round(3),
        'seasonal_wape_percent': (wape_comparison['seasonal_naive'] * 100).round(3),
        'absolute_gap_basis_points': ((wape_comparison['persistence'] - wape_comparison['seasonal_naive']).abs() * 10_000).round(2),
    }
).reset_index(drop=True)
display(comparator)

baseline_readiness = pd.Series(
    {
        'baseline run': manifest['run_id'],
        'development origins': forecasts['origin'].nunique(),
        'forecast horizons': sorted(forecasts['horizon'].unique().tolist()),
        'eligible forecast rows across both methods': len(forecasts),
        'prediction coverage': '100%',
        'frozen audit reproduced': True,
        'holdout performance opened': False,
        'candidate model evaluated': False,
    },
    name='value',
)
display(baseline_readiness.to_frame())

## Decision boundary

Baseline reproduction establishes the minimum evidence a candidate must beat; it does not justify a candidate class by itself. Review horizon, origin, state, bias, and volume-band behavior before freezing any feature set or candidate procedure. Holdout execution remains unavailable in this stage.